# Matched two-cycle modelling pipeline

This is the active Milestone 2 walkthrough. One chronological pipeline establishes simple baselines, tests controlled refinements, selects on development validation, freezes a threshold, and evaluates the saved recipes. This notebook reads evidence and scores saved pipelines; it does not fit models. The terminal cohort was previously inspected.


In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
root = Path.cwd().resolve()
if root.name == 'notebooks': root = root.parent
sys.path.insert(0, str(root))
run = root / 'artifacts/metrics/experiment-4/run-01'
protocol = json.loads((run / 'protocol.json').read_text())
selection = json.loads((run / 'selection_freeze.json').read_text())


## Problem and data preparation

Targets are delivery days and minimum-valid-review detractor status. Order-level aggregation precedes joins. Checkout inputs are reconstructed proxies. Chronological folds purge customers and enforce training-label maturity. Split summaries below describe fixed populations; they are not model-selection rules.


In [ ]:
pd.read_csv(run / 'split_audit.csv')


## Cycle 1 Simple references

Nine core features, original numeric inputs, plain unpenalized linear/logistic models, a single tree, a fixed forest, and dummy floors. Preprocessing fits within each training fold.


In [ ]:
summary = pd.read_csv(run / 'summary.csv')
summary.loc[summary.cycle == 1]


## Cycle 2 Controlled refinements

Task-specific expansion, input and regression-target log transformations, regularization, tree/forest comparisons, and training-only clipping are tested under identical task-specific fold memberships. Failed refinements remain visible.


In [ ]:
summary.loc[summary.cycle == 2]
pd.read_csv(run / 'paired_comparisons.csv')


## Selection and decision policy

Regression minimizes fold-mean MAE; classification maximizes fold-mean AP. Selection and the development OOF threshold freeze before terminal scoring. The selected classifier is LogisticRegression, not the historical forest.


In [ ]:
selection
policy = json.loads((run / 'decision_policy.json').read_text())
policy


## Final fit and diagnostics

Fold-training resubstitution, final-refit training, validation and terminal metrics are distinct. The terminal cohort is a previously inspected diagnostic. The experimental classifier policy has loss 9,902 versus no-alert loss 8,535, so it is not recommended for deployment.


In [ ]:
pd.read_csv(run / 'train_validation_terminal.csv')
terminal = pd.read_csv(run / 'terminal_metrics.csv')
terminal.loc[terminal.selected]


## Verified zero-fit scoring

The active configuration locates checksum-verified bundles. Predictions require constructed raw checkout features and order_id. Local scoring is not deployment approval.


In [ ]:
from src.models.experiment4_score import score
base = pd.read_csv(root / 'data/business/ml/orders_ml_features.csv', dtype={'order_id': 'string'})
sample = base.iloc[:5]
regression_scores = score('regression', sample)
classification_scores = score('classification', sample)
classification_scores


## Verification and reproducibility

Use experiments/experiment_4/README.md to reproduce into a fresh directory. Exact threshold reconstruction from CSV requires float_precision=round_trip. Original experiments and failed verification attempts remain preserved.


In [ ]:
json.loads((run / 'verification.json').read_text())
